# Variant effect on exon2 vs. insertion length/position

Boxplot: does the amount of random DNA inserted near a -150 promoter variant
change its predicted effect on exon2 (K562 RNA-seq, AlphaGenome)?

For every gene: exon2 VE = mean(ref_track - alt_track) over the exon2 span
(same offset in every condition, since exon2 never shifts -- see
`create_variant_all_promoters.py`). One VE value per gene per condition.

- x = insertion length (0 = baseline/no-insertion, else 25/50/75/100)
- y = exon2 variant effect
- color = condition (baseline / upstream -200bp / downstream -100bp insertion)
- each point = one gene

In [ ]:
import os
from concurrent.futures import ProcessPoolExecutor, as_completed

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import LinearSegmentedColormap, TwoSlopeNorm
from tqdm.auto import tqdm

In [ ]:
PROMOTER_DIR = "/scratch/st-cdeboer-1/sambina/position_mpra/outputs/8-aphagenome/all_k562_promoters"
# predictions_k562, not predictions/ (HepG2) -- see the matching note in
# create_variant_all_promoters.py. Plot outputs are saved alongside the data
# they're computed from rather than directly under PROMOTER_DIR, so this
# K562 run's plots can't collide with / overwrite the older HepG2 plots.
DEFAULT_PREDICTIONS_DIR = f"{PROMOTER_DIR}/predictions_k562"
DEFAULT_OUT = f"{DEFAULT_PREDICTIONS_DIR}/exon2_variant_effect_boxplot.svg"
DEFAULT_HEATMAP_OUT = f"{DEFAULT_PREDICTIONS_DIR}/exon2_variant_effect_heatmap.svg"
DEFAULT_CORR_OUT = f"{DEFAULT_PREDICTIONS_DIR}/exon2_ve_correlation_heatmap.svg"
DEFAULT_CORR_UPSTREAM_OUT = f"{DEFAULT_PREDICTIONS_DIR}/exon2_ve_correlation_heatmap_upstream.svg"
DEFAULT_SPEARMAN_OUT = f"{DEFAULT_PREDICTIONS_DIR}/exon2_ve_spearman_heatmap.svg"
DEFAULT_SPEARMAN_UPSTREAM_OUT = f"{DEFAULT_PREDICTIONS_DIR}/exon2_ve_spearman_heatmap_upstream.svg"
DEFAULT_SCATTER_OUT = f"{DEFAULT_PREDICTIONS_DIR}/exon2_ve_scatter_grid.svg"
DEFAULT_SCATTER_MATRIX_OUT = f"{DEFAULT_PREDICTIONS_DIR}/exon2_ve_scatter_matrix.svg"
DEFAULT_SCATTER_MATRIX_UPSTREAM_OUT = f"{DEFAULT_PREDICTIONS_DIR}/exon2_ve_scatter_matrix_upstream.svg"
DEFAULT_POSITION_LENGTH_OUT = f"{DEFAULT_PREDICTIONS_DIR}/exon2_ve_position_length_heatmap.svg"
DEFAULT_POSITION_LENGTH_SPEARMAN_OUT = (
    f"{DEFAULT_PREDICTIONS_DIR}/exon2_ve_position_length_heatmap_spearman.svg"
)

CONDITIONS = ["baseline", "upstream", "downstream"]
LENGTH_CATEGORIES = [25, 50, 75, 100]

# Matches CONDITION_KEYS in create_variant_all_promoters.py -- the 9 files
# ({condition_key}_ref.npy/_alt.npy) written by the new (all-lengths-per-gene)
# schema. compute_exon2_ve auto-detects which schema is actually on disk.
CONDITION_KEYS = ["baseline"] + [
    f"{position}_{k}" for position in ("upstream", "downstream") for k in LENGTH_CATEGORIES
]

# Categorical palette (dataviz skill default, slots 1/2/3 -- validated all-pairs CVD-safe)
COLORS = {
    "baseline": "#2a78d6",  # blue
    "upstream": "#eb6834",  # orange
    "downstream": "#1baf7a",  # aqua
}
LABELS = {
    "baseline": "No insertion (variant only)",
    "upstream": "Upstream insertion (-200)",
    "downstream": "Downstream insertion (-100)",
}

# Column order for the heatmap: no random sequence, then -100 (downstream), then -200 (upstream).
HEATMAP_CONDITIONS = ["baseline", "downstream", "upstream"]
HEATMAP_COLUMN_LABELS = ["None", "-100", "-200"]

# Diverging pair (dataviz skill default): blue <-> red, gray midpoint at 0.
DIVERGING_CMAP = LinearSegmentedColormap.from_list(
    "blue_gray_red", ["#2a78d6", "#f0efec", "#e34948"], N=256
)

# Sequential white->blue ramp for a 0-1 magnitude (R^2), matching plot_agarwal.ipynb.
SEQUENTIAL_BLUE_CMAP = LinearSegmentedColormap.from_list(
    "sequential_blue", ["#ffffff", "#3361A5"], N=256
)

## Parameters

Edit these paths as needed, then run the rest of the notebook.

In [ ]:
predictions_dir = DEFAULT_PREDICTIONS_DIR
out = DEFAULT_OUT
heatmap_out = DEFAULT_HEATMAP_OUT
corr_out = DEFAULT_CORR_OUT
corr_upstream_out = DEFAULT_CORR_UPSTREAM_OUT
spearman_out = DEFAULT_SPEARMAN_OUT
spearman_upstream_out = DEFAULT_SPEARMAN_UPSTREAM_OUT
scatter_out = DEFAULT_SCATTER_OUT
scatter_matrix_out = DEFAULT_SCATTER_MATRIX_OUT
scatter_matrix_upstream_out = DEFAULT_SCATTER_MATRIX_UPSTREAM_OUT
position_length_out = DEFAULT_POSITION_LENGTH_OUT
position_length_spearman_out = DEFAULT_POSITION_LENGTH_SPEARMAN_OUT
workers = None  # None -> len(os.sched_getaffinity(0)), i.e. CPUs allocated to this job/kernel

## Functions

In [ ]:
def _compute_condition_chunk(
    predictions_dir,
    condition_key,
    condition,
    fixed_length,
    gene_ids_chunk,
    starts_chunk,
    ends_chunk,
    lengths_chunk,
    row_start,
):
    """Runs in a worker process: computes VE for meta rows
    [row_start, row_start + len(gene_ids_chunk)) of one condition_key. Each
    worker reopens the memmap itself -- read-only mmaps are safely shared
    across processes, nothing is duplicated in memory up front."""
    ref = np.load(os.path.join(predictions_dir, f"{condition_key}_ref.npy"), mmap_mode="r")
    alt = np.load(os.path.join(predictions_dir, f"{condition_key}_alt.npy"), mmap_mode="r")

    out = []
    for local_idx, gene in enumerate(gene_ids_chunk):
        i = row_start + local_idx
        s, e = int(starts_chunk[local_idx]), int(ends_chunk[local_idx])
        ve = float(ref[i, s:e].astype(np.float32).mean() - alt[i, s:e].astype(np.float32).mean())
        length = fixed_length if fixed_length is not None else int(lengths_chunk[local_idx])
        out.append((gene, condition, length, ve))
    return out


def compute_exon2_ve(predictions_dir: str, workers: int = None) -> pd.DataFrame:
    """Reads whichever schema is actually in predictions_dir:
      - old (one length/gene): promoters_metadata.tsv has "length_category";
        files are {baseline,upstream,downstream}_{ref,alt}.npy.
      - new (all 4 lengths x both positions, every gene): metadata has
        insertion_seq_{25,50,75,100} instead; files are one per CONDITION_KEY
        (baseline, upstream_{K}, downstream_{K}).
    Either way, returns the same tidy columns: gene, condition
    (baseline/upstream/downstream), length (0/25/50/75/100), exon2_ve.

    Parallelized across processes: each condition is split into chunks of
    contiguous gene rows, and all (condition, chunk) tasks across all
    conditions run concurrently in a process pool -- this is what actually
    buys the speedup, since the per-row cost here is dominated by mmap I/O
    latency on network storage, and concurrent readers hide that latency
    instead of paying it one row at a time.
    """
    meta = pd.read_csv(os.path.join(predictions_dir, "promoters_metadata.tsv"), sep="\t")
    n = len(meta)
    old_schema = "length_category" in meta.columns
    condition_keys = CONDITIONS if old_schema else CONDITION_KEYS

    gene_ids = meta["gene"].to_numpy()
    starts = meta["exon2_start_offset"].to_numpy()
    ends = meta["exon2_end_offset"].to_numpy()
    length_categories = meta["length_category"].to_numpy() if old_schema else None

    workers = workers or len(os.sched_getaffinity(0))
    chunk_size = max(1, -(-n // workers))  # ~1 chunk per worker per condition

    tasks = []
    for condition_key in condition_keys:
        if condition_key == "baseline":
            condition, fixed_length = "baseline", 0
        elif old_schema:
            condition, fixed_length = condition_key, None  # length varies per gene
        else:
            condition, length_str = condition_key.rsplit("_", 1)
            fixed_length = int(length_str)

        for row_start in range(0, n, chunk_size):
            row_end = min(row_start + chunk_size, n)
            tasks.append(
                (
                    predictions_dir,
                    condition_key,
                    condition,
                    fixed_length,
                    gene_ids[row_start:row_end],
                    starts[row_start:row_end],
                    ends[row_start:row_end],
                    length_categories[row_start:row_end] if length_categories is not None else None,
                    row_start,
                )
            )

    rows = []
    with ProcessPoolExecutor(max_workers=workers) as pool:
        futures = [pool.submit(_compute_condition_chunk, *task) for task in tasks]
        for future in tqdm(as_completed(futures), total=len(futures), desc="chunks"):
            rows.extend(future.result())

    return pd.DataFrame(rows, columns=["gene", "condition", "length", "exon2_ve"])

In [ ]:
def plot(df: pd.DataFrame, out_path: str) -> None:
    fig, ax = plt.subplots(figsize=(9, 6))

    x_positions = {0: 0.0}
    x_positions.update({length: i + 1 for i, length in enumerate(LENGTH_CATEGORIES)})
    dodge = {"baseline": 0.0, "upstream": -0.15, "downstream": 0.15}

    rng = np.random.default_rng(0)
    for condition in CONDITIONS:
        sub = df[df["condition"] == condition]
        lengths = [0] if condition == "baseline" else LENGTH_CATEGORIES
        box_data, positions = [], []
        for length in lengths:
            vals = sub.loc[sub["length"] == length, "exon2_ve"].values
            if len(vals) == 0:
                continue
            box_data.append(vals)
            positions.append(x_positions[length] + dodge[condition])

        bp = ax.boxplot(
            box_data,
            positions=positions,
            widths=0.12,
            patch_artist=True,
            showfliers=False,
            boxprops=dict(facecolor=COLORS[condition], edgecolor=COLORS[condition], alpha=0.35),
            medianprops=dict(color=COLORS[condition], linewidth=2),
            whiskerprops=dict(color=COLORS[condition]),
            capprops=dict(color=COLORS[condition]),
        )
        for pos, vals in zip(positions, box_data):
            jitter = rng.uniform(-0.04, 0.04, size=len(vals))
            ax.scatter(
                np.full(len(vals), pos) + jitter,
                vals,
                s=8,
                color=COLORS[condition],
                alpha=0.35,
                linewidths=0,
                zorder=3,
            )

    ax.set_xticks([x_positions[l] for l in [0] + LENGTH_CATEGORIES])
    ax.set_xticklabels(["0\n(no insertion)"] + [str(l) for l in LENGTH_CATEGORIES])
    ax.set_xlabel("Random DNA inserted (bp)")
    ax.set_ylabel("Exon2 variant effect  (mean(ref - alt) over exon2, K562 RNA-seq)")
    ax.set_title("-150 promoter variant effect on exon2, by insertion position and length")
    ax.axhline(0, color="#898781", linewidth=1, zorder=0)

    handles = [
        plt.Line2D(
            [0],
            [0],
            marker="s",
            color="w",
            markerfacecolor=COLORS[c],
            markersize=10,
            label=LABELS[c],
        )
        for c in CONDITIONS
    ]
    ax.legend(handles=handles, loc="best", frameon=False)

    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

    fig.tight_layout()
    fig.savefig(out_path, dpi=150)
    print(f"Saved {out_path}")
    return fig

In [ ]:
def plot_heatmap(df: pd.DataFrame, out_path: str) -> None:
    """Small multiples: one heatmap per insertion length, rows = genes with that
    length category, columns = no random sequence / random at -100 / random at
    -200. Each gene has exactly one length_category, so the four panels
    partition all genes (they do not repeat a gene at multiple lengths)."""
    baseline_ve = df.loc[df["condition"] == "baseline"].set_index("gene")["exon2_ve"]

    panels = []
    for length in LENGTH_CATEGORIES:
        genes = sorted(df.loc[df["length"] == length, "gene"].unique())
        sub = df[(df["length"] == length) | (df["condition"] == "baseline")]
        sub = sub[sub["gene"].isin(genes)]
        pivot = sub.pivot_table(index="gene", columns="condition", values="exon2_ve")
        pivot = pivot.reindex(columns=HEATMAP_CONDITIONS)
        pivot = pivot.loc[baseline_ve.reindex(pivot.index).sort_values(ascending=False).index]
        panels.append((length, pivot))

    vmax = max(np.nanpercentile(np.abs(p.values), 99) for _, p in panels)
    vmax = max(vmax, 1e-12)
    norm = TwoSlopeNorm(vmin=-vmax, vcenter=0.0, vmax=vmax)

    fig, axes = plt.subplots(1, len(panels), figsize=(2.6 * len(panels), 8), sharey=False)
    im = None
    for ax, (length, pivot) in zip(axes, panels, strict=True):
        im = ax.imshow(
            pivot.values,
            aspect="auto",
            cmap=DIVERGING_CMAP,
            norm=norm,
            interpolation="nearest",
        )
        ax.set_xticks(range(len(HEATMAP_CONDITIONS)))
        ax.set_xticklabels(HEATMAP_COLUMN_LABELS, fontsize=8)
        ax.set_title(f"{length} bp insertion\n(n={len(pivot)} genes)", fontsize=10)
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)
    axes[0].set_ylabel("Genes (sorted by no-insertion effect)")

    fig.suptitle("Exon2 variant effect by insertion length and position", y=1.02)
    fig.text(
        0.5,
        -0.02,
        "Columns: no random sequence / random sequence inserted at -100 / at -200 (bp relative to TSS)",
        ha="center",
        fontsize=9,
        color="#52514e",
    )
    cbar = fig.colorbar(im, ax=axes, shrink=0.6, pad=0.02)
    cbar.set_label("Exon2 variant effect  (mean(ref - alt))")

    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved {out_path}")
    return fig

In [ ]:
def _correlation(x, y, method: str) -> float:
    """Pearson (default) or Spearman correlation. Spearman is just Pearson
    computed on ranks instead of raw values -- rank-transforming first makes
    it robust to outliers (a single extreme point can only move by one rank
    position, instead of contributing its full numeric deviation), at the
    cost of only capturing monotonic (not necessarily linear) relationships.
    Implemented via pandas .rank() rather than scipy to avoid the extra
    dependency."""
    if method == "spearman":
        x = pd.Series(x).rank().to_numpy()
        y = pd.Series(y).rank().to_numpy()
    return np.corrcoef(x, y)[0, 1]


def plot_correlation_heatmap(
    df: pd.DataFrame,
    out_path: str,
    condition: str,
    labels: list,
    axis_label: str,
    condition_label: str,
    method: str = "pearson",
) -> None:
    """Staircase heatmap of all-by-all correlation between exon2 VE at each
    insertion length for `condition`, plus the no-insertion baseline (length
    0). Rows are the 4 smaller-or-equal lengths (drops the largest, which
    never has anything smaller to pair with); columns are the 4
    larger-or-equal lengths (drops the smallest/baseline, which never has
    anything larger to pair with) -- a cell is only ever populated where the
    column's length exceeds the row's, so this omits the always-empty
    diagonal row/column a plain n x n lower-triangle grid would otherwise
    carry.

    Pearson (default) is shown as R^2 -- always >= 0, sequential blue scale.
    Spearman is shown as the signed rho itself, NOT squared -- squaring would
    erase the sign, and sign is exactly what a rank correlation is good for
    telling you (positive vs. inverse monotonic relationship). Diverging
    scale: negative rho renders red, positive blue, ~0 near-white."""
    lengths_order = [100, 75, 50, 25, 0]
    row_lengths, row_labels = lengths_order[1:], labels[1:]
    col_lengths, col_labels = lengths_order[:-1], labels[:-1]
    n_rows, n_cols = len(row_lengths), len(col_lengths)

    series = {}
    for length in lengths_order:
        cond = "baseline" if length == 0 else condition
        series[length] = df.loc[
            (df["condition"] == cond) & (df["length"] == length)
        ].set_index("gene")["exon2_ve"]

    value_matrix = np.full((n_rows, n_cols), np.nan)
    for i, row_len in enumerate(row_lengths):
        for j, col_len in enumerate(col_lengths):
            if col_len <= row_len:
                continue
            genes = series[row_len].index.intersection(series[col_len].index)
            x = series[row_len].reindex(genes).values
            y = series[col_len].reindex(genes).values
            r = _correlation(x, y, method)
            value_matrix[i, j] = r if method == "spearman" else r**2

    masked = np.ma.masked_invalid(value_matrix)

    fig, ax = plt.subplots(figsize=(1.2 * n_cols + 1, 1.2 * n_rows + 1))
    if method == "spearman":
        cmap = DIVERGING_CMAP.reversed()  # low(negative)=red, high(positive)=blue
        cmap.set_bad(color="none")
        norm = TwoSlopeNorm(vmin=-1, vcenter=0.0, vmax=1)
        im = ax.imshow(masked, cmap=cmap, norm=norm)
    else:
        cmap = SEQUENTIAL_BLUE_CMAP.copy()
        cmap.set_bad(color="none")
        im = ax.imshow(masked, cmap=cmap, vmin=0, vmax=1)

    for i in range(n_rows):
        for j in range(n_cols):
            val = value_matrix[i, j]
            if np.isnan(val):
                continue
            text_color = "white" if abs(val) > 0.6 else "#0b0b0b"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=9, color=text_color)

    ax.set_xticks(range(n_cols))
    ax.set_xticklabels(col_labels)
    ax.set_yticks(range(n_rows))
    ax.set_yticklabels(row_labels)
    ax.set_xlabel(axis_label)
    ax.set_ylabel(axis_label)
    method_label = "Spearman" if method == "spearman" else "Pearson"
    value_label = r"$\rho$" if method == "spearman" else "R$^2$"
    ax.set_title(
        f"Pairwise correlation of exon2 variant effect\n({method_label} {value_label}, {condition_label})"
    )

    for spine in ax.spines.values():
        spine.set_visible(False)

    cbar = fig.colorbar(im, ax=ax, shrink=0.9, pad=0.03)
    cbar.set_label(f"{method_label} {value_label}")

    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved {out_path}")
    return fig

In [ ]:
def plot_position_length_heatmap(df: pd.DataFrame, out_path: str, method: str = "pearson") -> None:
    """9x9 all-by-all correlation heatmap across all 9 conditions (baseline +
    4 upstream lengths + 4 downstream lengths). x-axis, left to right:
    upstream 100->25, baseline 0, downstream 25->100. y-axis, top to bottom:
    downstream 100->25, baseline 0, upstream 25->100 -- the mirror of the
    x-axis order. Because of that mirroring, a condition's self-correlation
    (trivially 1, masked out here same as elsewhere) falls on the
    ANTI-diagonal rather than the main diagonal: cells near the anti-diagonal
    are the most similar (nearby length and/or matching length at opposite
    position) pairs, cells in the far corners are the most different
    (e.g. top-left = downstream_100 vs upstream_100, opposite positions at
    the largest length each). This is the one plot that shows the length
    effect and the position effect together instead of one at a time."""
    x_seq = [("upstream", 100), ("upstream", 75), ("upstream", 50), ("upstream", 25)]
    x_seq += [("baseline", 0)]
    x_seq += [("downstream", 25), ("downstream", 50), ("downstream", 75), ("downstream", 100)]
    y_seq = list(reversed(x_seq))
    n = len(x_seq)
    tick_labels = [str(length) for _, length in x_seq]

    series = {}
    for key in x_seq:
        position, length = key
        series[key] = df.loc[
            (df["condition"] == position) & (df["length"] == length)
        ].set_index("gene")["exon2_ve"]

    value_matrix = np.full((n, n), np.nan)
    for i, y_key in enumerate(y_seq):
        for j, x_key in enumerate(x_seq):
            if y_key == x_key:
                continue
            genes = series[y_key].index.intersection(series[x_key].index)
            x = series[y_key].reindex(genes).values
            y = series[x_key].reindex(genes).values
            r = _correlation(x, y, method)
            value_matrix[i, j] = r if method == "spearman" else r**2

    masked = np.ma.masked_invalid(value_matrix)

    fig, ax = plt.subplots(figsize=(1.1 * n + 1, 1.1 * n + 1))
    if method == "spearman":
        cmap = DIVERGING_CMAP.reversed()
        cmap.set_bad(color="none")
        norm = TwoSlopeNorm(vmin=-1, vcenter=0.0, vmax=1)
        im = ax.imshow(masked, cmap=cmap, norm=norm)
    else:
        cmap = SEQUENTIAL_BLUE_CMAP.copy()
        cmap.set_bad(color="none")
        im = ax.imshow(masked, cmap=cmap, vmin=0, vmax=1)

    for i in range(n):
        for j in range(n):
            val = value_matrix[i, j]
            if np.isnan(val):
                continue
            text_color = "white" if abs(val) > 0.6 else "#0b0b0b"
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7, color=text_color)

    # Divider lines bounding the baseline row/column, to visually separate
    # the upstream block / baseline / downstream block.
    for boundary in (3.5, 4.5):
        ax.axvline(boundary, color="#c3c2b7", linewidth=1, zorder=3)
        ax.axhline(boundary, color="#c3c2b7", linewidth=1, zorder=3)

    ax.set_xticks(range(n))
    ax.set_xticklabels(tick_labels)
    ax.set_yticks(range(n))
    ax.set_yticklabels(tick_labels)
    ax.set_xlabel("←  Upstream random bases inserted (bp)      Downstream random bases inserted (bp)  →")
    ax.set_ylabel("←  Upstream random bases inserted (bp)      Downstream random bases inserted (bp)  →")

    method_label = "Spearman" if method == "spearman" else "Pearson"
    value_label = r"$\rho$" if method == "spearman" else "R$^2$"
    ax.set_title(
        f"Exon2 variant effect correlation across position and length\n({method_label} {value_label})"
    )

    for spine in ax.spines.values():
        spine.set_visible(False)

    cbar = fig.colorbar(im, ax=ax, shrink=0.9, pad=0.03)
    cbar.set_label(f"{method_label} {value_label}")

    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved {out_path}")
    return fig

In [ ]:
def plot_scatter_matrix(
    df: pd.DataFrame,
    out_path: str,
    condition: str,
    labels: list,
    axis_label: str,
    condition_label: str,
) -> None:
    """All-by-all scatterplot matrix: same strictly-lower-triangle layout as
    plot_correlation_heatmap (upper triangle + diagonal omitted), but each
    cell is the actual gene-by-gene scatter for that pair of insertion
    lengths instead of a single correlation value."""
    lengths_order = [100, 75, 50, 25, 0]
    n = len(lengths_order)

    series = {}
    for length in lengths_order:
        cond = "baseline" if length == 0 else condition
        series[length] = df.loc[
            (df["condition"] == cond) & (df["length"] == length)
        ].set_index("gene")["exon2_ve"]

    fig, axes = plt.subplots(n, n, figsize=(2.2 * n, 2.2 * n))
    for i, li in enumerate(lengths_order):
        for j, lj in enumerate(lengths_order):
            ax = axes[i, j]
            if j >= i:
                ax.axis("off")
                continue
            genes = series[li].index.intersection(series[lj].index)
            x = series[lj].reindex(genes).values
            y = series[li].reindex(genes).values
            ax.scatter(x, y, s=4, color=COLORS[condition], alpha=0.25, linewidths=0)
            for spine in ("top", "right"):
                ax.spines[spine].set_visible(False)
            ax.tick_params(labelsize=7)
            if i == n - 1:
                ax.set_xlabel(labels[j], fontsize=9)
            if j == 0:
                ax.set_ylabel(labels[i], fontsize=9)

    fig.suptitle(
        f"All-by-all exon2 variant effect ({condition_label})\n{axis_label}", y=1.01
    )
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved {out_path}")
    return fig

In [ ]:
def plot_scatter_grid(df: pd.DataFrame, out_path: str) -> None:
    """3x3 grid, one scatterplot per condition in CONDITION_KEYS: x = baseline
    (no-insertion) exon2 VE, y = that condition's exon2 VE, one point per
    gene. The baseline-vs-baseline panel is a trivial sanity check."""
    baseline_ve = df.loc[df["condition"] == "baseline"].set_index("gene")["exon2_ve"]

    panels = [("baseline", 0)] + [
        (position, k) for position in ("upstream", "downstream") for k in LENGTH_CATEGORIES
    ]

    fig, axes = plt.subplots(3, 3, figsize=(10, 10), sharex=True, sharey=True)
    for ax, (position, length) in zip(axes.flat, panels):
        sub = df[(df["condition"] == position) & (df["length"] == length)]
        x = baseline_ve.reindex(sub["gene"]).values
        y = sub["exon2_ve"].values
        r2 = np.corrcoef(x, y)[0, 1] ** 2

        ax.scatter(x, y, s=6, color=COLORS[position], alpha=0.3, linewidths=0)

        title = "No insertion (self)" if position == "baseline" else f"{LABELS[position]}, {length}bp"
        ax.set_title(f"{title}\nR$^2$={r2:.2f}", fontsize=9)
        for spine in ("top", "right"):
            ax.spines[spine].set_visible(False)

    for ax in axes[-1, :]:
        ax.set_xlabel("Baseline exon2 VE")
    for ax in axes[:, 0]:
        ax.set_ylabel("Condition exon2 VE")

    fig.suptitle("Exon2 variant effect: no-insertion baseline vs. each insertion condition", y=1.01)
    fig.tight_layout()
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    print(f"Saved {out_path}")
    return fig

## Run

In [ ]:
ve_path = os.path.join(os.path.dirname(out), "exon2_variant_effect.tsv")
if os.path.exists(ve_path):
    print(f"Found existing {ve_path}, skipping VE computation")
    df = pd.read_csv(ve_path, sep="\t")
else:
    df = compute_exon2_ve(predictions_dir, workers=workers)
    df.to_csv(ve_path, sep="\t", index=False)
df.head()

In [ ]:
plot(df, out)

In [ ]:
plot_heatmap(df, heatmap_out)

In [ ]:
downstream_lengths = [100, 75, 50, 25, 0]
downstream_labels = [str(-(150 + length)) for length in downstream_lengths]

plot_correlation_heatmap(
    df,
    corr_out,
    condition="downstream",
    labels=downstream_labels,
    axis_label="Variant position relative to TSS (bp)  [= -150 - insertion length]",
    condition_label="downstream insertion",
)

In [ ]:
plot_correlation_heatmap(
    df,
    spearman_out,
    condition="downstream",
    labels=downstream_labels,
    axis_label="Variant position relative to TSS (bp)  [= -150 - insertion length]",
    condition_label="downstream insertion",
    method="spearman",
)

In [ ]:
plot_scatter_matrix(
    df,
    scatter_matrix_out,
    condition="downstream",
    labels=downstream_labels,
    axis_label="Variant position relative to TSS (bp)  [= -150 - insertion length]",
    condition_label="downstream insertion",
)

In [ ]:
upstream_lengths = [100, 75, 50, 25, 0]
upstream_labels = [str(length) for length in upstream_lengths]

plot_correlation_heatmap(
    df,
    corr_upstream_out,
    condition="upstream",
    labels=upstream_labels,
    axis_label="Random sequence added upstream of the variant (bp)",
    condition_label="upstream insertion",
)

In [ ]:
plot_correlation_heatmap(
    df,
    spearman_upstream_out,
    condition="upstream",
    labels=upstream_labels,
    axis_label="Random sequence added upstream of the variant (bp)",
    condition_label="upstream insertion",
    method="spearman",
)

In [ ]:
plot_scatter_matrix(
    df,
    scatter_matrix_upstream_out,
    condition="upstream",
    labels=upstream_labels,
    axis_label="Random sequence added upstream of the variant (bp)",
    condition_label="upstream insertion",
)

In [ ]:
plot_scatter_grid(df, scatter_out)

In [ ]:
plot_position_length_heatmap(df, position_length_out)

In [ ]:
plot_position_length_heatmap(df, position_length_spearman_out, method="spearman")